In [1]:
import requests
import pandas as pd
import time

print("Environment Ready")

Environment Ready


In [2]:


subject = "self_help"

url = f"https://openlibrary.org/subjects/{subject}.json?limit=5"

response = requests.get(url)

print("Status:", response.status_code)

data = response.json()

print("Books found:", len(data.get("works", [])))

data["works"][0]

Status: 200
Books found: 5


{'key': '/works/OL276492W',
 'title': 'The Monk Who Sold His Ferrari',
 'edition_count': 58,
 'cover_id': 48817,
 'cover_edition_key': 'OL9245407M',
 'subject': ['Self-realization',
  'Self-actualization (psychology)',
  'Fiction, visionary & metaphysical',
  'Fiction',
  'Conduct of life',
  'Success',
  'Éxito',
  'Actualización de si mismo (Psicología)',
  'Fables',
  'Conducta (Ética)',
  'Fábulas',
  'Realización de sí mismo',
  'Ficción',
  'Succes',
  'Large type books',
  'Family',
  'Parenting',
  'Self help',
  'Spiritual life',
  'Life change events',
  'Réalisation de soi',
  'Romans, nouvelles',
  'Morale pratique',
  'Succès'],
 'ia_collection': ['americana',
  'delawarecountydistrictlibrary',
  'delawarecountydistrictlibrary-ol',
  'inlibrary',
  'internetarchivebooks',
  'library_of_atlantis',
  'openlibrary-d-ol',
  'popularchinesebooks',
  'printdisabled',
  'rochester-ol',
  'universityofthewest-ol'],
 'printdisabled': True,
 'lending_edition': 'OL31885519M',
 'lendi

In [3]:
import requests

work_key = "/works/OL276492W"

url = f"https://openlibrary.org{work_key}.json"

response = requests.get(url)

print("Status:", response.status_code)

work_data = response.json()

print(work_data.keys())

Status: 200
dict_keys(['first_publish_date', 'title', 'covers', 'first_sentence', 'key', 'authors', 'excerpts', 'type', 'subjects', 'description', 'latest_revision', 'revision', 'created', 'last_modified'])


In [4]:
work_data.get("description")

"Includes a bonus excerpt of Robin Sharma's upcoming The Secret Letters of the Monk Who Sold His Ferrari.\r\n\r\nWith more than four million copies sold in fifty-one languages, The Monk Who Sold His Ferrari launched a bestselling series and continues to help people from every walk of life live with far greater success, happiness and meaning in these times of dramatic uncertainty.\r\n\r\nThe Monk Who Sold His Ferrari celebrates the story of Julian Mantle, a successful but misguided lawyer whose physical and emotional collapse propels him to confront his life. The result is an engaging odyssey on how to release your potential and live with passion, purpose and peace.\r\n\r\nA brilliant blend of timeless wisdom and cutting-edge success principles, The Monk Who Sold His Ferrari is now, more than ever, a guide for the times, as countless Canadians dedicate themselves to living a life where family, work and personal fulfillment are achieved in harmonious balance."

In [5]:
def get_book_details(work_key):

    url = f"https://openlibrary.org{work_key}.json"

    response = requests.get(url)

    if response.status_code != 200:
        return None

    data = response.json()

    title = data.get("title", "")

    description = data.get("description", "")

    if isinstance(description, dict):
        description = description.get("value", "")

    subjects = data.get("subjects", [])

    if not title:
        return None

    return {
        "title": title,
        "description": description,
        "subjects": subjects
    }

In [6]:
book = get_book_details("/works/OL276492W")

book

{'title': 'The Monk Who Sold His Ferrari',
 'description': "Includes a bonus excerpt of Robin Sharma's upcoming The Secret Letters of the Monk Who Sold His Ferrari.\r\n\r\nWith more than four million copies sold in fifty-one languages, The Monk Who Sold His Ferrari launched a bestselling series and continues to help people from every walk of life live with far greater success, happiness and meaning in these times of dramatic uncertainty.\r\n\r\nThe Monk Who Sold His Ferrari celebrates the story of Julian Mantle, a successful but misguided lawyer whose physical and emotional collapse propels him to confront his life. The result is an engaging odyssey on how to release your potential and live with passion, purpose and peace.\r\n\r\nA brilliant blend of timeless wisdom and cutting-edge success principles, The Monk Who Sold His Ferrari is now, more than ever, a guide for the times, as countless Canadians dedicate themselves to living a life where family, work and personal fulfillment are a

In [7]:
import requests

subjects = [
    "self_help",
    "psychology",
    "motivation",
    "mindfulness",
    "habits",
    "leadership",
    "communication",
    "creativity",
    "productivity",
    "emotions"
]

work_keys = []

for subject in subjects:
    url = f"https://openlibrary.org/subjects/{subject}.json?limit=100"
    res = requests.get(url)
    data = res.json()

    for work in data.get("works", []):
        key = work.get("key")
        if key:
            work_keys.append(key)

len(work_keys)

969

In [8]:
import time
import pandas as pd

books = []
seen = set()

for key in work_keys:

    if key in seen:
        continue

    seen.add(key)

    data = get_book_details(key)

    if data is None:
        continue

    # filtro de calidad
    if len(data["description"]) < 80:
        continue

    books.append({
        "title": data["title"],
        "description": data["description"],
        "subjects": data["subjects"],
        "source": "api"
    })

    if len(books) % 50 == 0:
        print("Collected:", len(books))

    if len(books) >= 700:
        break

    time.sleep(0.2)

df_api = pd.DataFrame(books)

df_api.shape

Collected: 50
Collected: 100
Collected: 150
Collected: 200
Collected: 250
Collected: 300
Collected: 350
Collected: 400


(425, 4)

In [11]:
import requests

queries = [
    "self help",
    "psychology motivation",
    "mindfulness habits",
    "success productivity",
    "emotional intelligence",
    "leadership communication",
    "creativity thinking",
    "stress anxiety",
]

scraped_keys = set()

for q in queries:
    url = f"https://openlibrary.org/search.json?q={q}&limit=100"

    res = requests.get(url)
    data = res.json()

    docs = data.get("docs", [])

    for d in docs:
        key = d.get("key")
        if key:
            scraped_keys.add(key)

    print(q, "->", len(scraped_keys))

len(scraped_keys)

self help -> 100
psychology motivation -> 200
mindfulness habits -> 298
success productivity -> 398
emotional intelligence -> 498
leadership communication -> 598
creativity thinking -> 698
stress anxiety -> 795


795

In [15]:
import requests
import time

def get_book_details(work_key):

    url = f"https://openlibrary.org{work_key}.json"

    try:
        res = requests.get(url, timeout=10)

        if res.status_code != 200:
            return None

        data = res.json()

        title = data.get("title", "")
        if not title:
            return None

        description = data.get("description", "")
        if isinstance(description, dict):
            description = description.get("value", "")
        if description is None:
            description = ""

        subjects = data.get("subjects", [])
        if subjects is None:
            subjects = []

        return {
            "title": title,
            "description": description,   # puede estar vacío, está OK
            "subjects": subjects,
            "source": "scraping"
        }

    except:
        return None

In [16]:
print("kernel alive")import requests

url = "https://openlibrary.org/works/OL276492W.json"

res = requests.get(url, timeout=10)

print(res.status_code)
print(res.json()["title"])

kernel alive


In [17]:
import requests

url = "https://openlibrary.org/works/OL276492W.json"

res = requests.get(url, timeout=10)

print(res.status_code)
print(res.json()["title"])

200
The Monk Who Sold His Ferrari


In [18]:
scraped_books = []
seen = set()

In [19]:
import time
import requests

def get_book_details(work_key):

    url = f"https://openlibrary.org{work_key}.json"

    try:
        res = requests.get(url, timeout=10)

        if res.status_code != 200:
            return None

        data = res.json()

        title = data.get("title", "")
        if not title:
            return None

        description = data.get("description", "")
        if isinstance(description, dict):
            description = description.get("value", "")
        if description is None:
            description = ""

        subjects = data.get("subjects", [])
        if subjects is None:
            subjects = []

        return {
            "title": title,
            "description": description,
            "subjects": subjects,
            "source": "scraping"
        }

    except:
        return None

In [20]:
for i in range(5):
    print("loop running", i)

loop running 0
loop running 1
loop running 2
loop running 3
loop running 4


In [21]:
print("keys variable exists:", "scraped_keys" in globals())

if "scraped_keys" in globals():
    print("length:", len(scraped_keys))
    print(list(scraped_keys)[:10])

keys variable exists: True
length: 795
['/works/OL37886311W', '/works/OL13304571M', '/works/OL29598068W', '/works/OL3740735W', '/works/OL31180516W', '/works/OL17476520W', '/works/OL24208866W', '/works/OL40608633W', '/works/OL20515761W', '/works/OL3407921W']


In [22]:
for i, key in enumerate(list(scraped_keys)[:10]):
    print("processing:", key)

processing: /works/OL37886311W
processing: /works/OL13304571M
processing: /works/OL29598068W
processing: /works/OL3740735W
processing: /works/OL31180516W
processing: /works/OL17476520W
processing: /works/OL24208866W
processing: /works/OL40608633W
processing: /works/OL20515761W
processing: /works/OL3407921W


In [23]:
import time

scraped_books = []
seen = set()

for i, key in enumerate(scraped_keys):

    if key in seen:
        continue

    seen.add(key)

    data = get_book_details(key)

    if data is None:
        continue

    # filtro razonable (NO agresivo)
    if len(data["title"]) < 1:
        continue

    scraped_books.append({
        "title": data["title"],
        "description": data["description"],
        "subjects": data["subjects"],
        "source": "scraping"
    })

    if len(scraped_books) % 50 == 0:
        print("Collected:", len(scraped_books))

    if len(scraped_books) >= 700:
        break

    time.sleep(0.2)

import pandas as pd
df_scraping = pd.DataFrame(scraped_books)
df_scraping.shape

Collected: 50
Collected: 100
Collected: 150
Collected: 200
Collected: 250
Collected: 300
Collected: 350
Collected: 400
Collected: 450
Collected: 500
Collected: 550
Collected: 600
Collected: 650
Collected: 700


(700, 4)

In [24]:
df_final = pd.concat([df_api, df_scraping], ignore_index=True)

df_final.drop_duplicates(subset=["title"], inplace=True)

df_final.shape

(1013, 4)

In [25]:
df_final["content"] = (
    df_final["title"].fillna("") + " " +
    df_final["description"].fillna("") + " " +
    df_final["subjects"].astype(str)
)

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df_final["content"])

In [27]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [28]:
import numpy as np

indices = pd.Series(df_final.index, index=df_final["title"]).drop_duplicates()

def recommend(title, top_n=5):

    if title not in indices:
        return "Book not found"

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:top_n+1]

    book_idx = [i[0] for i in sim_scores]

    return df_final.iloc[book_idx][["title", "description"]]

In [29]:
df_final["title"].head(10)

0           The Monk Who Sold His Ferrari
1                              The Secret
2           The Power of Focused Thinking
3                                  嫌われる勇気
4    The Definitive Book of Body Language
5           God and the evolving universe
6                              Girlosophy
7                       How to Do Nothing
8                       How to pass exams
9                     The Twelfth Insight
Name: title, dtype: object

In [30]:
recommend("The Monk Who Sold His Ferrari")

,title,description
272,Family Wisdom from the Monk Who Sold His Ferrari,"Richard Carlson, author of the worldwide bests..."
889,Self Help,
752,The Power of Positive Thinking,"In this phenomenal bestseller, “written with t..."
242,How to Win Friends and Influence People,Available for the first time ever in trade pap...
697,Renewal,


In [31]:
random_title = df_final["title"].sample(1).values[0]
print(random_title)
recommend(random_title)

Motivation, emotion, and cognition


,title,description
602,Understanding motivation and emotion,
1076,The psychology of motivation,
524,The psychology of learning and motivation,
162,"Attention, representation, and human performance","""This volume presents a rare occasion where sc..."
626,Human motivation and emotion,


In [32]:
emotion_to_subjects = {
    "happy": ["self help", "creativity"],
    "sad": ["psychology", "self help"],
    "stressed": ["mindfulness", "habits"],
    "motivated": ["productivity", "leadership"],
    "lost": ["philosophy", "self help"],
    "anxious": ["emotions", "mindfulness"]
}

In [33]:
df_final["subjects_clean"] = df_final["subjects"].astype(str).str.lower()

In [34]:
def recommend_by_emotion(emotion):

    subjects = emotion_to_subjects.get(emotion)

    if not subjects:
        return "Emotion not supported"

    # filtro por subjects
    filtered = df_final[
        df_final["subjects_clean"].apply(
            lambda x: any(s in x for s in subjects)
        )
    ]

    if len(filtered) == 0:
        return "No books found for this emotion"

    # ranking usando tu modelo TF-IDF
    idxs = filtered.index
    scores = cosine_sim[idxs].mean(axis=1)

    filtered = filtered.copy()
    filtered["score"] = scores

    return filtered.sort_values("score", ascending=False)[
        ["title", "description"]
    ].head(5)

In [35]:
recommend_by_emotion("stressed")

,title,description
222,Mental efficiency,The mind can only be conquered by regular medi...
179,The mindful twenty-something,"Stress is a modern-day epidemic, and with the ..."
169,Mindful Being (Alchemy of Love Mindfulness Tra...,Mindful Being Personal Development Course is a...
171,Control Your Mind and Master Your Feelings,We oftentimes look towards the outside world t...
192,Mindful Leaders Leading Self - a Heideggerian ...,The purpose of this study is to explore mindfu...


In [36]:
emotion_map = {
    "sad": ["mindfulness", "self-help", "spiritual", "healing", "emotion"],
    "anxious": ["mindfulness", "stress", "anxiety", "calm", "meditation"],
    "stressed": ["productivity", "habits", "focus", "time", "mindfulness"],
    "low_motivation": ["motivation", "success", "productivity", "habits"],
    "angry": ["mindfulness", "emotional intelligence", "self-control"],
    "confused": ["decision", "thinking", "clarity", "psychology"],
}

In [37]:
df_final["text"] = (
    df_final["title"].fillna("") + " " +
    df_final["description"].fillna("") + " " +
    df_final["subjects"].astype(str)
).str.lower()

In [46]:
valid_domains = [
    "self-help", "psychology", "mindfulness",
    "habit", "emotion", "motivation",
    "productivity", "stress", "anxiety",
    "spiritual", "personal development"
]

In [47]:
def recommend_by_emotion(emotion, df, top_n=10):

    keywords = emotion_map.get(emotion, [])

    def score(text):
        text = text.lower()

        emotion_score = sum(1 for k in keywords if k in text)
        domain_score_val = sum(1 for d in valid_domains if d in text)

        return emotion_score * 2 + domain_score_val

    df["score"] = df["text"].apply(score)

    result = df[df["score"] > 0]

    return result.sort_values("score", ascending=False)[
        ["title", "description", "score"]
    ].head(top_n)

In [48]:
recommend_by_emotion("stressed", df_final)

,title,description,score
171,Control Your Mind and Master Your Feelings,We oftentimes look towards the outside world t...,13
848,The here-and-now habit,"""Bad habits can take a hefty toll on your heal...",12
238,The Dopamine Code Workbook,Reshape your brain--and break the Dopamine Cod...,12
176,Attending,"""The first book for the general public about m...",11
185,Happy & Present in 2-Minutes,**Unlock Real Freedom from Stress & Anxiety – ...,11
195,Why Your Mind Won't Let You Rest,"<p class=""p1"" style=""margin:0px 0px 10px;font-...",11
237,The Dopamine Code,The Dopamine Code is a game-changer for brains...,11
189,Selah,A child's death evokes intense and long-lastin...,10
199,Focus,"In Focus, Daniel Goleman delves into the scien...",10
169,Mindful Being (Alchemy of Love Mindfulness Tra...,Mindful Being Personal Development Course is a...,10


In [51]:
df_final.to_csv("books.csv", index=False)

In [52]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\vicky\BookRecommendationProject\notebooks
['.ipynb_checkpoints', '01_data_collection.ipynb', '02_data_audit.ipynb', 'books.csv', 'google_books', 'Untitled.ipynb']


In [53]:
print(df.columns)

NameError: name 'df' is not defined

FileNotFoundError: [Errno 2] No such file or directory: 'books.csv'